In [1]:
import torch
import numpy as np
import pandas as pd

from helpers.persistence import *
from helpers.openml_data_v2 import openml_cc18_list
from helpers.openml_data import tabular_id_list
from helpers.progress_bar import ProgressBar
from itertools import product
import helpers.trainer_multiworker_cluster_v2 as trainer

In [2]:
import random

def set_random_seeds():
    torch.manual_seed(0)
    np.random.seed(0)
    random.seed(0) # mcar sampler uses random maybe   
    
set_random_seeds()

In [3]:
if torch.cuda.is_available():
    device = "cuda:0"
else:
    device = "cpu"

trainer.device = device
print(device)

cuda:0


In [4]:
# expt = 'test-10-scarf-patient-3clusterReplace'
expt = 'test-10-scarf-patient-3clusterReplaceTest'

save_path, export_path = f'./saved_vars/{expt}.pkl', f'./exports/{expt}.csv'

dataset_results = load_var(save_path) or {}
# dataset_results = {}
# len(dataset_results)

Could not load ./saved_vars/test-10-scarf-patient-3clusterReplaceTest.pkl because [Errno 2] No such file or directory: './saved_vars/test-10-scarf-patient-3clusterReplaceTest.pkl'


In [5]:

names = openml_cc18_list[:1]
names = tabular_id_list[10:20]

from helpers.openml_data_v2 import hard_list
names = hard_list
names = [ 1063,  1510,  1464,   469,  
         458,  1494,  1068,  1049,    
         23, 1050, 40975, 40982,  
         1067,  1487,  1485,  4134,
         40701,  1497, 1475,  4538][::2] 

# names = names[:5]
names = [1510]

corruptions = [
    'knn-replace',
    'knn-rf',
    'knn-cutmix',
    'cluster-replace',
    'mixup-draw',
    'mixup-draw-features',
    
]

n_pairs = [128]
missing_rate = 0.6
missing_type = 'mcar'
patience = 9999

# losses = ['simclr','scarf','bad-scarf']
losses = ['simclr']

expt_list = list(product(names, corruptions, losses))
pbar = ProgressBar(expt_list)
trainer.pbar = pbar

for dataset_name, c, loss in pbar:     
    pbar.clear_prefix()
    
    trainer.settings['corruptor2'].update({
        'method': c,
        'corruption_rate': 0.6,
    })
    trainer.settings.update({
        'method': 'bootstrap', 
        'pretrain_loss':loss,
        'corrupt_before_ohe': True,
        'test_pretrain': False,
        'patience': patience,
        'bootstrap': 30,
        'train_workers': 0,
    })
    
    # print(trainer.settings)
    
    corruption_key = f"{c}-{missing_type}-{missing_rate}" if c in ['knn', 'mice'] else c
    key = f'{dataset_name}_{corruption_key}_loss={loss}_patience={patience}'
    expt1 = f'{expt}_{key}'
    
    pbar.add_prefix(key)

    if key in dataset_results.keys():
        # print(key, 'done')
        continue
        
    pbar.set_description('')

    # print(key, 'doing')
    set_random_seeds()
    scores, z_scores, p_scores, pretrain_time = trainer.do_all(dataset_name, expt=expt1)

    dataset_results[key] = {
        'scores': scores,
        'z_scores': z_scores,
        'p_scores': p_scores,
        'pretrain_time': pretrain_time,
    }
    save_var(dataset_results, save_path)


1510_cluster-replace_loss=simclr_patience=9999 | fold=30/30 |  Finetuning | epoch: 200/200;mean_loss: 0.00

test accuracy: 0.9707  ( 0.0162 )
time taken, total=3612.23s or 120.41s per fold


1510_cluster-replace_loss=simclr_patience=9999 | fold=30/30 |  Finetuning | epoch: 200/200;mean_loss: 0.00


In [6]:
cols = [ 'dataset', 'corruption', 'npairs', 'patience',
        'fold', 'test_score', 
        'z_score_LR', 'p_score_LR', 
        'z_score_GBT', 'p_score_GBT', 
        'avg_loss_time']
rows = []

for k,v in dataset_results.items():
    common_data = k.split('_')
    
    # common_data[-1] = np.round(float(common_data[-1]), 5)
    n_folds = len(v['scores'])
    
    for i in range(n_folds):
        avg_time = np.mean(v['pretrain_time'][i])
        base_scores = [
            v['z_scores']['Logistic Regression'][i], 
            v['p_scores']['Logistic Regression'][i], 
            v['z_scores']['Gradient Boosting'][i], 
            v['p_scores']['Gradient Boosting'][i]
        ] if len(v['z_scores']['Logistic Regression'])>0 else [0,0,0,0]
        row = common_data[:-1] + [int(common_data[-1].split('=')[1])] + [i, v['scores'][i], 
                             *base_scores, 
                             avg_time]
        rows.append(row)
        # print(row)
        # break
        
df = pd.DataFrame(rows, columns=cols)
df.to_csv(export_path)

In [7]:
tmp = df.groupby(['dataset', 'corruption'])[[
    'test_score', 
    'z_score_LR', 'p_score_LR', 
    'z_score_GBT', 'p_score_GBT', 
    'avg_loss_time']].agg('mean')
tmp

,,test_score,z_score_LR,p_score_LR,z_score_GBT,p_score_GBT,avg_loss_time
dataset,corruption,,,,,,
1510,cluster-replace,0.97073,0.0,0.0,0.0,0.0,0.003076
